# Worked example S1: one temperature node by hand

This notebook reproduces, step by step, the state estimated for rice husk at 400 °C (Supplementary Section S1). Every intermediate quantity is computed explicitly and compared with the result of the package.

In [ ]:
import os
import shutil
import sys
import tempfile
from pathlib import Path
ARTICLE = Path.cwd().parent / "results"
OUTPUT = Path.cwd() / "output" / "01_worked_example_single_node"
OUTPUT.mkdir(parents=True, exist_ok=True)
if (ARTICLE / "best_vectors.npz").exists():
    handle, temporary = tempfile.mkstemp(dir=OUTPUT, suffix=".npz")
    os.close(handle)
    shutil.copyfile(ARTICLE / "best_vectors.npz", temporary)
    os.replace(temporary, OUTPUT / "best_vectors.npz")
os.environ.setdefault("CCPYRO_RESULTS", str(OUTPUT))
os.environ.setdefault("CCPYRO_FIGDATA", str(OUTPUT / "figdata"))
try:
    import ccpyro  # installed with `pip install -e .`
except ImportError:  # running from a clone without installation
    sys.path.insert(0, str(Path.cwd().parent / "src"))
    import ccpyro
import numpy as np
import pandas as pd
np.set_printoptions(precision=6, suppress=True)
print("ccpyro", ccpyro.__version__)

In [ ]:
from ccpyro import model as mp
feed = mp.load_feedstocks()["rice husk"]
catalogue, _ = mp.load_gas_data()
curve, runs = mp.mean_thermogram("rice husk")
thermal = mp.build_thermal_basis(feed, curve, runs)
gas = catalogue[("rice husk", "pelletized")]
ctx = mp.ModelContext(feedstock=feed, thermal=thermal, composition=gas.mean, composition_sd=gas.sd,
                      alpha=mp.ALPHA_TRANSFERRED)
with np.load(mp.BEST_VECTORS) as archive:                     # reported estimate
    vector = archive["rice husk (pelletized)"]
s = mp.unpack(vector, ctx)
k = 1                                                          # node at 400 C
M = mp.M

## Step 1. Parent formula, Eqs. (1)-(3)

In [ ]:
c = feed.daf_composition
x0 = (c["H"] / M["H"]) / (c["C"] / M["C"])
y0 = (c["O"] / M["O"]) / (c["C"] / M["C"])
M_bio = M["C"] + M["H"] * x0 + M["O"] * y0
print(f"x0 = {x0:.4f}, y0 = {y0:.4f}, M_bio = {M_bio:.3f} g/mol C")
assert np.isclose(x0, ctx.x0) and np.isclose(y0, ctx.y0)

## Step 2. Thermal basis

In [ ]:
print(f"theta(400 C) = {ctx.theta[k]:.4f} +/- {ctx.theta_scale[k]:.4f}; FC_daf = {feed.fixed_carbon_daf:.4f}")
print(f"T_M = {ctx.peak:.1f} C, T_on = {thermal.onset_temperature:.1f} C, beta = {ctx.beta:.5f} 1/C, alpha = {ctx.alpha}")

## Step 3. Decision variables at the node

In [ ]:
for name in mp.VARIABLES:
    print(f"{name:6s} = {getattr(s, name)[k]:.6f}")
kappa, A, a_H, a_O = s.parameters
print(f"kappa_W = {kappa:.4f}, A = {A:.4f}, a_H = {a_H:.4f}, a_O = {a_O:.4f}")

## Step 4. Closed-form elimination of the balances, Eqs. (8)-(10)

In [ ]:
n_t = 1 - s.n_c[k] - s.n_CO[k] - s.n_CO2[k] - s.n_CH4[k]
n_W = y0 - s.y_c[k] * s.n_c[k] - s.y_t[k] * n_t - s.n_CO[k] - 2 * s.n_CO2[k]
n_H2 = 0.5 * (x0 - s.x_c[k] * s.n_c[k] - s.x_t[k] * n_t - 2 * n_W - 4 * s.n_CH4[k])
print(f"n_t = {n_t:.6f}, n_W = {n_W:.6f}, n_H2 = {n_H2:.6f}")
assert np.allclose([n_t, n_W, n_H2], [s.n_t[k], s.n_H2O[k], s.n_H2[k]])
# the balances of Eqs. (5)-(7) close exactly
carbon = s.n_c[k] + n_t + s.n_CO[k] + s.n_CO2[k] + s.n_CH4[k]
hydrogen = s.x_c[k] * s.n_c[k] + s.x_t[k] * n_t + 2 * n_W + 4 * s.n_CH4[k] + 2 * n_H2
oxygen = s.y_c[k] * s.n_c[k] + s.y_t[k] * n_t + n_W + s.n_CO[k] + 2 * s.n_CO2[k]
print("balance residuals:", 1 - carbon, x0 - hydrogen, y0 - oxygen)

## Step 5. Mass fractions, Eqs. (11)-(14)

In [ ]:
Mc = M["C"] + M["H"] * s.x_c[k] + M["O"] * s.y_c[k]
Mt = M["C"] + M["H"] * s.x_t[k] + M["O"] * s.y_t[k]
w = {"char": s.n_c[k] * Mc / M_bio, "tar": n_t * Mt / M_bio, "water": n_W * mp.MM["H2O"] / M_bio,
     "gas": (s.n_CO[k] * mp.MM["CO"] + s.n_CO2[k] * mp.MM["CO2"] + s.n_CH4[k] * mp.MM["CH4"] + n_H2 * mp.MM["H2"]) / M_bio}
print({key: round(float(v), 5) for key, v in w.items()}, "sum =", round(float(sum(w.values())), 12))

## Step 6. Gas released over the stage, Eq. (17)

In [ ]:
released = np.array([s.n_CO[k] - s.n_CO[k-1], s.n_CO2[k] - s.n_CO2[k-1], s.n_CH4[k] - s.n_CH4[k-1], s.n_H2[k] - s.n_H2[k-1]])
phi = released / released.sum()
print("model   :", phi.round(4))
print("measured:", ctx.composition[k].round(4))

## Step 7. Closures

In [ ]:
kprime = kappa * mp.MM["CO2"] / mp.MM["H2O"]
print(f"water share {n_W / (n_W + s.n_CO2[k]):.4f} vs closure {kprime / (1 + kprime):.4f}")
target = mp.tar_selectivity(mp.T_NODES, A, ctx.alpha, ctx.beta, ctx.peak)[k]
print(f"tar selectivity {w['tar'] / (w['tar'] + w['gas']):.4f} vs Fraser-Suzuki {target:.4f}")
print(f"x_c {s.x_c[k]:.4f} vs x0*theta^aH {x0 * ctx.theta[k]**a_H:.4f}; y_c {s.y_c[k]:.4f} vs y0*theta^aO {y0 * ctx.theta[k]**a_O:.4f}")

## Step 8. Admissibility, Eqs. (27)-(31)

In [ ]:
g = mp.constraints(vector, ctx)
print(f"smallest constraint value over the grid: {g.min():.2e}")
print(f"x_t - x_c = {s.x_t[k] - s.x_c[k]:.4f}, y_t - y_c = {s.y_t[k] - s.y_c[k]:.4f}")

## Step 9. Residuals and objective

In [ ]:
terms = mp.objective_terms(vector, ctx)
print({key: round(float(v), 4) for key, v in terms.items()})
print("E =", round(sum(terms.values()) / mp.OBJECTIVE_SCALE, 5))